# || Feature Engineering : Bureau Dataset ||

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder
print("All libraries imported!")
pd.set_option('display.float_format', lambda x: '%.3f' % x)

All libraries imported!


In [3]:
bureau=pd.read_csv('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/raw/bureau.csv')
bureau

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.000,-153.000,NaN,0,91323.000,0.000,NaN,0.000,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.000,NaN,NaN,0,225000.000,171342.000,NaN,0.000,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.000,NaN,NaN,0,464323.500,NaN,NaN,0.000,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.000,NaN,NaN,0.000,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.000,NaN,77674.500,0,2700000.000,NaN,NaN,0.000,Consumer credit,-21,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1716423,259355,5057750,Active,currency 1,-44,0,-30.000,NaN,0.000,0,11250.000,11250.000,0.000,0.000,Microloan,-19,NaN
1716424,100044,5057754,Closed,currency 1,-2648,0,-2433.000,-2493.000,5476.500,0,38130.840,0.000,0.000,0.000,Consumer credit,-2493,NaN
1716425,100044,5057762,Closed,currency 1,-1809,0,-1628.000,-970.000,NaN,0,15570.000,NaN,NaN,0.000,Consumer credit,-967,NaN
1716426,246829,5057770,Closed,currency 1,-1878,0,-1513.000,-1513.000,NaN,0,36000.000,0.000,0.000,0.000,Consumer credit,-1508,NaN


# || Data Cleaning and Processing ||

### Missing Values

In [4]:
# We know from EDA, these 7 have missing values
missing_col=['AMT_ANNUITY','AMT_CREDIT_MAX_OVERDUE','DAYS_ENDDATE_FACT','AMT_CREDIT_SUM_LIMIT','AMT_CREDIT_SUM_DEBT','DAYS_CREDIT_ENDDATE','AMT_CREDIT_SUM']
# Lets handle them one by one

#### Handling amt_annuity, this is the monthly payment amount that borrower credits in the loan amount.

In [5]:
print(bureau['AMT_ANNUITY'].describe())
print(f"The median : {bureau['AMT_ANNUITY'].median()}")

count      489637.000
mean        15712.758
std        325826.949
min             0.000
25%             0.000
50%             0.000
75%         13500.000
max     118453423.500
Name: AMT_ANNUITY, dtype: float64
The median : 0.0


As seen in the describe also, 50% of the values are 0. And the top 25% values are in range : 13500-118453423.

In [6]:
# For loans that have already ended will have reamining days credit end date in negative or zero.
bureau[['AMT_ANNUITY','AMT_CREDIT_SUM']]
bureau[bureau['AMT_CREDIT_SUM_DEBT']>0][['AMT_CREDIT_SUM','AMT_CREDIT_SUM_DEBT','AMT_ANNUITY','DAYS_CREDIT','DAYS_CREDIT_ENDDATE']]

,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_ANNUITY,DAYS_CREDIT,DAYS_CREDIT_ENDDATE
1,225000.000,171342.000,NaN,-208,1075.000
5,180000.000,71017.380,NaN,-273,27460.000
6,42103.800,42103.800,NaN,-43,79.000
13,89910.000,76905.000,NaN,-96,269.000
17,450000.000,520920.000,NaN,-381,NaN
...,...,...,...,...,...
1716403,1431000.000,1092226.500,NaN,-765,1061.000
1716404,450000.000,432787.500,NaN,-99,997.000
1716412,225000.000,10971.000,NaN,-541,7.000
1716417,67500.000,2466.000,NaN,-740,31128.000


Since 71 % of the value is missing, creating a isMissing column will be useful while keeping the actual values. Another thing is, to get a general idea of a borrower's repayment ability, which is EMI/NMI ratio, the rough annuity can be calculated as (loan paid till now)/(days_credit) * 30.

In [7]:
# imputing credit category wise median into annuity 
category_medians = bureau.groupby('CREDIT_TYPE')['AMT_ANNUITY'].transform('median')
bureau['b_annuity_imp'] = bureau['AMT_ANNUITY'].fillna(category_medians).fillna(0)

# creating expected annuity feature for loans 
debt_paid = bureau['AMT_CREDIT_SUM'] - bureau['AMT_CREDIT_SUM_DEBT']
closed_days = bureau['DAYS_ENDDATE_FACT'] - bureau['DAYS_CREDIT']
active_days = np.abs(bureau['DAYS_CREDIT']) + 1
days_elapsed = np.where(
    bureau['CREDIT_ACTIVE'] == 'Closed', 
    closed_days, 
    active_days
)
days_elapsed = np.nan_to_num(days_elapsed, nan=1.0)
days_elapsed = np.maximum(days_elapsed, 1)
bureau['b_exp_annuity']=(debt_paid/days_elapsed)*30.4     # avg of all 12 months
bureau['b_exp_annuity']=bureau['b_exp_annuity'].fillna(0)
# is missing binary feature 
bureau['b_annuity_missing'] = bureau['AMT_ANNUITY'].isna().astype('int8')

# repayment intensity feature
bureau['b_annuity_loan_ratio'] = bureau['b_annuity_imp'] / (bureau['AMT_CREDIT_SUM'] + 1).fillna(0)


4 new features : isMissing binary, expected annuity, annuity to loan ratio and imputed annuity were added. Expected annuity gives information about possible EMI of the loan. The imputed annuity is the median annuity of the given credit type. All these can be used to estimate repayment capability. We are keeping the annuity feature with the left over NaN.

In [8]:

bureau[bureau['AMT_ANNUITY']>0][['AMT_ANNUITY','b_annuity_imp','b_exp_annuity','b_annuity_missing','b_annuity_loan_ratio']]

,AMT_ANNUITY,b_annuity_imp,b_exp_annuity,b_annuity_missing,b_annuity_loan_ratio
769,2691.000,2691.000,3821.229,0,0.060
781,24462.000,24462.000,4488.919,0,0.202
787,8181.000,8181.000,9318.801,0,0.073
789,8181.000,8181.000,15174.040,0,0.092
790,8061.210,8061.210,0.000,0,8061.210
...,...,...,...,...,...
1716274,43735.500,43735.500,66.287,0,0.076
1716275,43735.500,43735.500,1560.235,0,2.785
1716285,94378.500,94378.500,20962.534,0,0.459
1716290,58554.000,58554.000,37377.049,0,0.065


#### Handling amt_credit_max_overdue : 
This contains maximum overdue amount in CB loan so far. The loans that have never been overdue, will have NaN.

In [9]:
bureau['AMT_CREDIT_MAX_OVERDUE'].isna().mean()*100
# 65 % values are missing in this column.

np.float64(65.51326359159837)

In [10]:
bureau['AMT_CREDIT_MAX_OVERDUE'].describe()

count      591940.000
mean         3825.418
std        206031.606
min             0.000
25%             0.000
50%             0.000
75%             0.000
max     115987185.000
Name: AMT_CREDIT_MAX_OVERDUE, dtype: float64

In [11]:
bureau.groupby(
    'CREDIT_TYPE'
)['AMT_CREDIT_MAX_OVERDUE'].mean()

CREDIT_TYPE
Another type of loan                             6304.571
Car loan                                        21776.492
Cash loan (non-earmarked)                        1486.651
Consumer credit                                  3487.544
Credit card                                      1728.850
Interbank credit                                      NaN
Loan for business development                  108698.345
Loan for purchase of shares (margin lending)          NaN
Loan for the purchase of equipment              62437.500
Loan for working capital replenishment           5769.459
Microloan                                         791.523
Mobile operator loan                                0.000
Mortgage                                        99725.731
Real estate loan                                  459.000
Unknown type of loan                            17429.358
Name: AMT_CREDIT_MAX_OVERDUE, dtype: float64

The useful features that can be obtained from this column are :
1. b_max_overdue_missing : binary column that gives if the information missing or not.
2. b_overdue_credit : ratio of overdue amount with the total credit given to the borrower. 
3. b_overdue_debt : ratio of overdue with the loan outstanding.
4. b_current_max_overdue : ratio of current amt overdue to the max overdue.
5. b_max_overdue_x_curr_dpd : current dpd X max overdue. If both values large, resultant is even larger. Borrower defaults big and longer.

In [12]:
# Max overdue missing or not
bureau['b_max_overdue_missing']=bureau['AMT_CREDIT_MAX_OVERDUE'].isna().astype(np.int8)

# Max overdue relative to total credit amount
bureau['b_max_overdue_to_credit'] =    bureau['AMT_CREDIT_MAX_OVERDUE']/bureau['AMT_CREDIT_SUM'].replace(0, np.nan)


# Max overdue relative to outstanding debt
bureau['b_max_overdue_to_debt'] = (
    bureau['AMT_CREDIT_MAX_OVERDUE'] /
    bureau['AMT_CREDIT_SUM_DEBT'].replace(0, np.nan)
)

# Current overdue relative to historical maximum overdue
bureau['b_current_to_max_overdue'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    bureau['AMT_CREDIT_MAX_OVERDUE'].replace(0, np.nan)
)

# Historical severity × current delinquency
bureau['b_max_overdue_x_curr_dpd'] = (
    bureau['AMT_CREDIT_MAX_OVERDUE'] *
    bureau['CREDIT_DAY_OVERDUE']
)

#### Now the 'amt_credit_sum' :
This is the loan amount given to the borrower. The features that can be the most beneficial are :
1. b_paid_to_days : the amount of loan paid to the no of days loan has been active.
2. b_left_to_full : the ratio of loan yet to be paid.
3. b_loan_to_tenure : the total amount of credit/total tenure in days of the loan.
4. b_curr_overdue_to_loan : the ratio of current overdue to total credit.
5. b_credit_x_curr_dpd
6. b_credit_x_prolonged

In [13]:
# -----------------------------------
# AMT_CREDIT_SUM Derived Features
# -----------------------------------

# Helper variables
credit_age = np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)

total_tenure = np.abs(
    bureau['DAYS_CREDIT_ENDDATE'] -
    bureau['DAYS_CREDIT']
).replace(0, np.nan)

amount_paid = (
    bureau['AMT_CREDIT_SUM'] -
    bureau['AMT_CREDIT_SUM_DEBT']
)

# 1. Amount paid per day since loan started
bureau['b_paid_to_days'] = (
    amount_paid /
    credit_age
)

# 2. Fraction of loan still unpaid
bureau['b_left_to_full'] = (
    bureau['AMT_CREDIT_SUM_DEBT'] /
    bureau['AMT_CREDIT_SUM'].replace(0, np.nan)
)

# 3. Credit amount per day of loan tenure
bureau['b_loan_to_tenure'] = (
    bureau['AMT_CREDIT_SUM'] /
    total_tenure
)

# 4. Current overdue relative to total credit
bureau['b_curr_overdue_to_loan'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    bureau['AMT_CREDIT_SUM'].replace(0, np.nan)
)

# 5. Credit exposure × current overdue days
bureau['b_credit_x_curr_dpd'] = (
    bureau['AMT_CREDIT_SUM'] *
    bureau['CREDIT_DAY_OVERDUE']
)

# 6. Credit exposure × number of prolongations
bureau['b_credit_x_prolonged'] = (
    bureau['AMT_CREDIT_SUM'] *
    bureau['CNT_CREDIT_PROLONG']
)

#### Now for loan outstanding : 'amt_credit_sum_debt'

In [14]:
# Missing flag
bureau['b_debt_missing'] = (
    bureau['AMT_CREDIT_SUM_DEBT']
    .isna()
    .astype(np.int8)
)

# Debt burden relative to total credit
bureau['b_debt_to_credit'] =bureau['AMT_CREDIT_SUM_DEBT'] /bureau['AMT_CREDIT_SUM'].replace(0, np.nan)


# Debt burden relative to loan age
bureau['b_debt_to_days'] = bureau['AMT_CREDIT_SUM_DEBT'] /np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)

# Debt burden relative to total tenure
bureau['b_debt_to_tenure'] = (
    bureau['AMT_CREDIT_SUM_DEBT'] /
    np.abs(
        bureau['DAYS_CREDIT_ENDDATE'] -
        bureau['DAYS_CREDIT']
    ).replace(0, np.nan)
)

# Current overdue relative to debt
bureau['b_curr_overdue_to_debt'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    bureau['AMT_CREDIT_SUM_DEBT'].replace(0, np.nan)
)

# Historical max overdue relative to debt
bureau['b_max_overdue_to_debt'] = (
    bureau['AMT_CREDIT_MAX_OVERDUE'] /
    bureau['AMT_CREDIT_SUM_DEBT'].replace(0, np.nan)
)

# Debt exposure × current DPD
bureau['b_debt_x_curr_dpd'] = (
    bureau['AMT_CREDIT_SUM_DEBT'] *
    bureau['CREDIT_DAY_OVERDUE']
)

# Debt exposure × prolongations
bureau['b_debt_x_prolonged'] = (
    bureau['AMT_CREDIT_SUM_DEBT'] *
    bureau['CNT_CREDIT_PROLONG']
)

# Status Interaction Features

bureau['b_active_debt'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Active',
    bureau['AMT_CREDIT_SUM_DEBT'],
    0
)

bureau['b_closed_debt'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Closed',
    bureau['AMT_CREDIT_SUM_DEBT'],
    0
)

bureau['b_sold_debt'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Sold',
    bureau['AMT_CREDIT_SUM_DEBT'],
    0
)

bureau['b_bad_debt_debt'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Bad debt',
    bureau['AMT_CREDIT_SUM_DEBT'],
    0
)

#### Now for current overdue amount : 'amt_credit_sum_overdue'

In [15]:
# -----------------------------------
# AMT_CREDIT_SUM_OVERDUE Features
# -----------------------------------

# Missing flag
bureau['b_overdue_missing'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE']
    .isna()
    .astype(np.int8)
)

# Has current overdue
bureau['b_has_overdue'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] > 0
).astype(np.int8)

# -----------------------------------
# Relative Overdue Features
# -----------------------------------

# Current overdue relative to total credit
bureau['b_overdue_to_credit'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    bureau['AMT_CREDIT_SUM'].replace(0, np.nan)
)

# Current overdue relative to outstanding debt
bureau['b_overdue_to_debt'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    bureau['AMT_CREDIT_SUM_DEBT'].replace(0, np.nan)
)

# Current overdue relative to historical max overdue
bureau['b_overdue_to_max_overdue'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    bureau['AMT_CREDIT_MAX_OVERDUE'].replace(0, np.nan)
)

# -----------------------------------
# Time-Based Features
# -----------------------------------

# Overdue amount per day of loan age
bureau['b_overdue_to_days'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)
)

# Overdue amount per day of loan tenure
bureau['b_overdue_to_tenure'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    np.abs(
        bureau['DAYS_CREDIT_ENDDATE']
        - bureau['DAYS_CREDIT']
    ).replace(0, np.nan)
)

# -----------------------------------
# Interaction Features
# -----------------------------------

# Overdue amount × overdue days
bureau['b_overdue_x_curr_dpd'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] *
    bureau['CREDIT_DAY_OVERDUE']
)

# Overdue amount × prolongations
bureau['b_overdue_x_prolonged'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] *
    bureau['CNT_CREDIT_PROLONG']
)

# -----------------------------------
# Status Interaction Features
# -----------------------------------

bureau['b_active_overdue'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Active',
    bureau['AMT_CREDIT_SUM_OVERDUE'],
    0
)

bureau['b_closed_overdue'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Closed',
    bureau['AMT_CREDIT_SUM_OVERDUE'],
    0
)

bureau['b_sold_overdue'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Sold',
    bureau['AMT_CREDIT_SUM_OVERDUE'],
    0
)

bureau['b_bad_debt_overdue'] = np.where(
    bureau['CREDIT_ACTIVE'] == 'Bad debt',
    bureau['AMT_CREDIT_SUM_OVERDUE'],
    0
)

#### Now day_overdue and prolonged :

In [16]:
# -----------------------------------
# CREDIT_DAY_OVERDUE Features
# -----------------------------------

# Missing flag
bureau['b_curr_dpd_missing'] = (
    bureau['CREDIT_DAY_OVERDUE']
    .isna()
    .astype(np.int8)
)

# Any current delinquency
bureau['b_has_curr_dpd'] = (
    bureau['CREDIT_DAY_OVERDUE'] > 0
).astype(np.int8)

# Delinquency severity flags
bureau['b_dpd30_flag'] = (
    bureau['CREDIT_DAY_OVERDUE'] >= 30
).astype(np.int8)

bureau['b_dpd60_flag'] = (
    bureau['CREDIT_DAY_OVERDUE'] >= 60
).astype(np.int8)

bureau['b_dpd90_flag'] = (
    bureau['CREDIT_DAY_OVERDUE'] >= 90
).astype(np.int8)

# Log transformed DPD
bureau['b_log_curr_dpd'] = np.log1p(
    bureau['CREDIT_DAY_OVERDUE']
)

# -----------------------------------
# CNT_CREDIT_PROLONG Features
# -----------------------------------

# Missing flag
bureau['b_prolong_missing'] = (
    bureau['CNT_CREDIT_PROLONG']
    .isna()
    .astype(np.int8)
)

# Has ever been prolonged
bureau['b_has_prolong'] = (
    bureau['CNT_CREDIT_PROLONG'] > 0
).astype(np.int8)

# Multiple prolongations
bureau['b_multi_prolong'] = (
    bureau['CNT_CREDIT_PROLONG'] >= 2
).astype(np.int8)

# Prolongations × current delinquency
bureau['b_prolong_x_curr_dpd'] = (
    bureau['CNT_CREDIT_PROLONG'] *
    bureau['CREDIT_DAY_OVERDUE']
)

# Prolongations × current overdue amount
bureau['b_prolong_x_overdue'] = (
    bureau['CNT_CREDIT_PROLONG'] *
    bureau['AMT_CREDIT_SUM_OVERDUE']
)


We have not created ratio features from prolonged as most of the values are 0.

#### Now the time based features : days and end dates

In [17]:
# -----------------------------------
# Time-Based Bureau Features
# -----------------------------------

# Loan age (days since loan originated)
bureau['b_credit_age'] = np.abs(
    bureau['DAYS_CREDIT']
)

# Remaining time until expected closure
bureau['b_remaining_term'] = (
    bureau['DAYS_CREDIT_ENDDATE']
)

# Original loan tenure
bureau['b_total_tenure'] = np.abs(
    bureau['DAYS_CREDIT_ENDDATE']
    - bureau['DAYS_CREDIT']
)

# Actual completed tenure
bureau['b_actual_tenure'] = np.abs(
    bureau['DAYS_ENDDATE_FACT']
    - bureau['DAYS_CREDIT']
)

# Difference between expected and actual closure
bureau['b_tenure_gap'] = (
    bureau['DAYS_ENDDATE_FACT']
    - bureau['DAYS_CREDIT_ENDDATE']
)

# Bureau recency
bureau['b_update_recency'] = np.abs(
    bureau['DAYS_CREDIT_UPDATE']
)

# Time since last bureau update relative to age
bureau['b_update_to_age'] = (
    np.abs(bureau['DAYS_CREDIT_UPDATE']) /
    np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)
)

# -----------------------------------
# Timing Flags
# -----------------------------------

# Expected end date already passed
bureau['b_expired_loan'] = (
    bureau['DAYS_CREDIT_ENDDATE'] < 0
).astype(np.int8)

# Loan closed
bureau['b_has_actual_end'] = (
    bureau['DAYS_ENDDATE_FACT'].notna()
).astype(np.int8)

# Closed later than expected
bureau['b_closed_late'] = (
    bureau['DAYS_ENDDATE_FACT']
    >
    bureau['DAYS_CREDIT_ENDDATE']
).astype(np.int8)

# Closed earlier than expected
bureau['b_closed_early'] = (
    bureau['DAYS_ENDDATE_FACT']
    <
    bureau['DAYS_CREDIT_ENDDATE']
).astype(np.int8)

# Recently opened loans
bureau['b_recent_30d'] = (
    bureau['DAYS_CREDIT'] >= -30
).astype(np.int8)

bureau['b_recent_90d'] = (
    bureau['DAYS_CREDIT'] >= -90
).astype(np.int8)

bureau['b_recent_180d'] = (
    bureau['DAYS_CREDIT'] >= -180
).astype(np.int8)

bureau['b_recent_365d'] = (
    bureau['DAYS_CREDIT'] >= -365
).astype(np.int8)

# -----------------------------------
# Time × Exposure Features
# -----------------------------------

# Credit intensity per day
bureau['b_credit_per_day'] = (
    bureau['AMT_CREDIT_SUM'] /
    np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)
)

# Debt intensity per day
bureau['b_debt_per_day'] = (
    bureau['AMT_CREDIT_SUM_DEBT'] /
    np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)
)

# Overdue intensity per day
bureau['b_overdue_per_day'] = (
    bureau['AMT_CREDIT_SUM_OVERDUE'] /
    np.abs(bureau['DAYS_CREDIT']).replace(0, np.nan)
)

In [18]:
bureau.shape

(1716428, 85)

In [19]:
for col in bureau.columns:
    print(col)

SK_ID_CURR
SK_ID_BUREAU
CREDIT_ACTIVE
CREDIT_CURRENCY
DAYS_CREDIT
CREDIT_DAY_OVERDUE
DAYS_CREDIT_ENDDATE
DAYS_ENDDATE_FACT
AMT_CREDIT_MAX_OVERDUE
CNT_CREDIT_PROLONG
AMT_CREDIT_SUM
AMT_CREDIT_SUM_DEBT
AMT_CREDIT_SUM_LIMIT
AMT_CREDIT_SUM_OVERDUE
CREDIT_TYPE
DAYS_CREDIT_UPDATE
AMT_ANNUITY
b_annuity_imp
b_exp_annuity
b_annuity_missing
b_annuity_loan_ratio
b_max_overdue_missing
b_max_overdue_to_credit
b_max_overdue_to_debt
b_current_to_max_overdue
b_max_overdue_x_curr_dpd
b_paid_to_days
b_left_to_full
b_loan_to_tenure
b_curr_overdue_to_loan
b_credit_x_curr_dpd
b_credit_x_prolonged
b_debt_missing
b_debt_to_credit
b_debt_to_days
b_debt_to_tenure
b_curr_overdue_to_debt
b_debt_x_curr_dpd
b_debt_x_prolonged
b_active_debt
b_closed_debt
b_sold_debt
b_bad_debt_debt
b_overdue_missing
b_has_overdue
b_overdue_to_credit
b_overdue_to_debt
b_overdue_to_max_overdue
b_overdue_to_days
b_overdue_to_tenure
b_overdue_x_curr_dpd
b_overdue_x_prolonged
b_active_overdue
b_closed_overdue
b_sold_overdue
b_bad_debt_o

In [21]:
bureau.shape

(1716428, 85)

In [20]:
bureau.to_csv('/Users/rageshwer/Goal ML/Projects/Home_Credit_Default_Risk/data/interim/bureau_fe.csv')